Vamos a usar caracteres como si fuesen tokens 

Encode: Para un nombre de una ciudad me devuelve una lista de los índices de cada uno de los caracteres correspondientes. 
Decode: Recibe la lista y devuelve un string vacío en donde va agregando cosa itos. 

Ahora vamos a encodear el texto en una varianle data; va a ser una lista gigante de números. (long es un tipo de entero de torch)

Dividimos entre validation (se utiliza para ver como responde el modelo sobre datos que no vió) y training (entreno el modelo, hago foward-pass, calculo la función de pérdida, modifico los gradientes, etc). Validation sirve por ejemplo para ver cuando terminar de entrenar. 

Cuando dejamos de entrenar, ahí empezamos a ver como le va al modelo con los datos de testing. Ese es el valor que se tiene en cuenta. 

Se define un n, donde el 90% es de training, y el resto de validation. 

batch: tiene context size, batch size, elementos del dataset y la etiqueta (y)

Hay que definir los hiperparámetros de nuestro modelo. Son cosas que se definen antes de entrenarlo, pero se pueden ir cambiando. 

Dependiendo de cada valor que elegimos se van a terminar usando diferentes matrices. 

El context size indica diferentes contextos tiene nuestro modelo. 

Como y va a usar el mismo x pero corrido un lugar. 

- context_size: cantidad máxima de tokens que el modelo mira hacia atrás
- embedding size: dimensión del vector con que se representa cada token
- n_hidden: cantidad de neuronas de la capa oculta de la red feed-forward (MLP) dentro de cada bloque. Suele ser `4 * n_embd`.
- att_head_size: dimensión de los vectores query, key y value de cada cabeza de atención. Normalmente `n_embd // num_heads`.
- num_heads: cuántas cabezas de atención corren en paralelo en cada bloque. Cada una puede fijarse en relaciones distintas entre tokens, y al final sus salidas se concatenan.
- num_blocks (n_layer)**: cuántos bloques transformer (attention + feed-forward) se apilan. Más bloques da un modelo más profundo y con más capacidad.


- dropout: probabilidad de apagar neuronas al azar durante el entrenamiento, para regularizar y evitar overfitting (por ejemplo, 0.2 = 20%).
- batch_size: cuántas secuencias independientes se procesan juntas en cada paso de entrenamiento. Es la `B` en `(B, T, C)`.
- steps (max_iters): cantidad total de iteraciones de entrenamiento, es decir, cuántas veces se actualizan los pesos.
- eval_interval: cada cuántos steps se frena para medir el loss en train y validación.
- learning_rate: tamaño del paso con que el optimizador (por ejemplo, AdamW) actualiza los pesos. Si es muy alto el entrenamiento diverge, y si es muy bajo aprende lento.
- eval_iters: cuántos batches se promedian para estimar el loss en cada evaluación. Así el valor sale menos ruidoso que con un solo batch.

Como no codeamos nada de encoding, no hace falta la parte de multi-head attention. Vamos a hacer un GPT que es sólo generador. Nos enfocamos en masker multi- head attenion y feed forward

Tenemos una clase Linear que representa la capa feed forward. 

Recibe parámetro device para usar torch; para usar idealmente una GPU. 
Pone las matrices de bias y de pesos con ese device. 

Es un arreglo de perceptrones. 

Tenemos una clase tanh, para calcular la tangente hiperbólica de x. 

Tenemos una clase embeddings. 
Dado un índice, la fila que corresponde a ese embedding. Se logra con self.out = self.w[x]  (Las filas que corresponden a w en x). 

Como los pesos tiene que ver con la función de pérdida y se van ajustando con el entrenamiento, va a ir cambiando la matriz. 

Tenemso una clase sequential. 
Para cada layer de la lista de capas, se hace x = layer(x), hace forward pass para una lista de capas. 

Hasta ahora vimos lo necesario para las capas Feed Linear y Masked Multi-Head.

Ahora nos centramos en las capas Add & Norm.

Los modelos con muchas capas no se podrían entrenar si no fuesen por la normalización. Hace que los resultados de la capa no sean muy chicos (transfiriendo gradientes por números chicos) o muy chicos (achatar los gradientes). 

No hay modelo de lenguaje posible sin estas capas.

Formas de hacerlo:
 - Normalizar los batches (que sean similares los samples)
 - Normalizar por capas (se usa en transformers, por eso tenemos una clase LayerNorm)

Tenemos una clase SelfAttentionHead
Se inicializa dando el tamaño de head y el tamaño de los embeddings. 

Hacemos la traspuesta sobre la dimensión de contexto, dejando la de batch igual. 

Hacemos softmax 

(Es todo lo de la clase Attention)

Preguntas parcial: 

¿Qué es self-attention? Son Masked Multi-Head y Multi-Head Attention.Las queries, keys y values salen todas de la misma secuencia. Los tokens se prestan atención entre ellos mismos. La entrada es solo x. 

¿Qué es cross-attention? Las queries vienen de una secuencia y las keys y values de otra. 

La única entrada de la función call de SelfAttention es x. 

Tenemos una clase MultiHeadAttention: 
Es una lista de SelfAttentionHead. 

Necesita una capa linear que hace una proyección. Si concateno las salidas de los attention head, me sirvan como entrada para el próximo transformer block. 

Tenemos una clase FeedFoward (es de feature_size x feature_size*4)

Son dos capas lineales 

Tenemos una clase TransformerDedodeBlock

Tenemos como capas: un multi-head attention, un feed-foward y dos capas de normalización. 

Al resultado de aplicarle Multi-Head Attention a x y normalizada, le sumamos otra vez la x (es un residuo)

En el gráfico hace post-normalization, es decir: primero normalizo la entrada y luego se lo paso al mecanismo de atención.

En el código se hace pre-normalization, es decir: primero aplico el mecanismo de atención y luego normalizo la salida. 

Ya tenemos todos los componentes para hacer el transformer.
Usamos el NameGenerator:
    - Tengo 1 matriz de embeddings
    - Tengo 1 matriz para cada posición, que embedding le corresponde
    - Tengo 3 transformerDecoderBlock 

El embedding es la suma de elemento a elemento de emb y pos. 
Esto ingresa a cada transformer. 

En stochastic gradient decent. Tomamos un batch que ausmimos representativo. Calculamos función de perdida, backwards, modificamos W en una fracción del gradiente de cada uno de los pesos. El learning rate es igual para todos. 

Lo que hace ADAMW, es que no hace que los learning rate sea igual para todos los w. Esto implicia llevar la cuenta de que tan rápido hay que modificar cada w. No solo vamos a modificar la gradiente de cada w, si no, también hay que ver que tan distinto es de la vez anterior. 